# RNA Seq Pipeline
#### Author: Kylie Gregory 
#### Created On: 11/25/2026
#### Last Modified: 11/25/2026
#### Purpose: Create an RNA-seq pipeline to determine what gene expression changes are consistent across two independent datasets comparing dilated cardiomyopathy (DCM) and healthy (non-failing donor) groups. 

#### Files: 


In [1]:
#load in libraries

library(DESeq2)
library(tidyverse)
library(GEOquery)

Loading required package: S4Vectors



Loading required package: stats4



Loading required package: BiocGenerics



Loading required package: generics




Attaching package: ‘generics’




The following objects are masked from ‘package:base’:

    as.difftime, as.factor, as.ordered, intersect, is.element, setdiff,
    setequal, union




... (19 outputs left out)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     


── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ lubridate::%within%() masks IRanges::%within%()
✖ dplyr::collapse()     masks IRanges::collapse()
✖ dplyr::combine()      masks Biobase::combine(), BiocGenerics::combine()
✖ dplyr::count()        masks matrixStats::count()
✖ dplyr::desc()         masks IRanges::desc()
✖ tidyr::expand()       masks S4Vectors::expand()
✖ dplyr::filter()       masks stats::filter()
✖ dplyr::first()        masks S4Vectors::first()
✖ dplyr::lag()          masks stats::lag()
✖ ggplot2::Position()   masks BiocGenerics::Position(), base::Position()
✖ purrr::reduce()       masks GenomicRanges::reduce(), IRanges::reduce()
✖ dplyr::rename()       masks S4Vectors::rename()
✖ lubridate::second()   masks S4Vectors::second()
✖ lubridate::second<-() masks S4Vectors::second<-()
✖ dplyr::slice()        masks IRanges::slice()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


Setting options('download.file.method.GEOquery'='auto')



Setting options('GEOquery.inmemory.gpl'=FALSE)



In [2]:
#read in RNA-seq raw counts dataset


create_raw_counts_table <- function(raw_counts_url, GEO_number){
    #either GSE141910.tsv.gz or GSE116250.tsv.gz
    file_name <- paste(GEO_number, ".tsv.gz", sep="")
    download.file(raw_counts_url, file_name)
    count_table <- read.table(file_name, header=T,sep="\t")
    #make GeneID column the unique row identifiers 
    count_table <- column_to_rownames(count_table, var ="GeneID")

    return(count_table)
    
}

In [6]:

#load in column data to map each sample to a condition 

create_column_data <- function(GEO_number){
                               
    #get the GEO series                          
    gse <- getGEO(GEO_number)
                               
    #get the phenotypic data about each sample                           
    sample_data <- pData(gse[[1]])
    
     #the condition (DCM vs non-failing) is in different columns for the two datasets we are analysing, so created an if/else block 
                               
    #this bloc filters to only DCM vs non-failing samples, selects only that column, changes the name of the column to condition, and edits the exact wording of non-failing and DCM to be consistent among the datasets to allow downstream code to be modular
                              
                 
    if (GEO_number =="GSE141910")
        {
        col_data <- sample_data %>% filter(characteristics_ch1 %in% c("etiology: Non-Failing Donor",
                                                                    "etiology: Dilated cardiomyopathy (DCM)")) %>% select(characteristics_ch1) %>% rename(condition=characteristics_ch1)
        
        col_data$condition <- sub("etiology: ", "", col_data$condition)
            
    } else{
        col_data <-  col_data <- sample_data %>% filter(`disease:ch1` %in% c("non-failing", "dilated cardiomyopathy")) %>% select(`disease:ch1`) %>% rename(condition=`disease:ch1`)
        
        col_data$condition <- sub("non-failing", "Non-Failing Donor", col_data$condition)
        col_data$condition <- sub("dilated cardiomyopathy", "Dilated cardiomyopathy (DCM)", 
                                  col_data$condition)
    
    }
    
    
    return (col_data)

}



#Link for working with GEOquery: https://bioconductor.org/packages/release/bioc/vignettes/GEOquery/inst/doc/GEOquery.html#why-geoquery
                               



In [7]:
d1_geo_number <- "GSE141910"
d2_geo_number <- "GSE116250"

d1_count_url <- "https://www.ncbi.nlm.nih.gov/geo/download/?type=rnaseq_counts&acc=GSE141910&format=file&file=GSE141910_raw_counts_GRCh38.p13_NCBI.tsv.gz"
d2_count_url <- "https://www.ncbi.nlm.nih.gov/geo/download/?type=rnaseq_counts&acc=GSE116250&format=file&file=GSE116250_raw_counts_GRCh38.p13_NCBI.tsv.gz"

#get each datasets raw counts matrix
d1_count_table <- create_raw_counts_table(d1_count_url, d1_geo_number)
d2_count_table <- create_raw_counts_table(d2_count_url, d2_geo_number)

#get each datasets condition information 
d1_col_data <- create_column_data(d1_geo_number)
d2_col_data <- create_column_data(d2_geo_number)
    

Found 1 file(s)



GSE141910_series_matrix.txt.gz



Using locally cached version: /tmp/Rtmppa19k2/GSE141910_series_matrix.txt.gz



Using locally cached version of GPL16791 found here:
/tmp/Rtmppa19k2/GPL16791.soft.gz 



Found 1 file(s)



GSE116250_series_matrix.txt.gz



Using locally cached version: /tmp/Rtmppa19k2/GSE116250_series_matrix.txt.gz



Using locally cached version of GPL16791 found here:
/tmp/Rtmppa19k2/GPL16791.soft.gz 



In [9]:
#count matrix has all of the samples, so need to filter to just get DCM and non-failing donor 
#for DESeq2, need rows of col_data to match columns of count_tale


create_filtered_tables <- function(col_data, count_table){
    
    #col_data contains only sample numbers belonging to DCM and non failing donors 
    col_data_samples <- rownames(col_data)
    
    #count_table contains sample numbers from every condition 
    all_samples <- colnames(count_table)
    
    #get overlapping sample numbers 
    dcm_nf_samples <- intersect(col_data_samples, all_samples)
    
    #new count table contains only DCM and nonfailing donors that we also have informatation about in col_data
    count_table_filtered <- count_table %>%
    select(all_of(dcm_nf_samples))
    
    #new col_data contains only DCM and nonfailing donors that
    #were also available in the filtered count table
    
    col_data_filtered <- col_data %>% filter(rownames(col_data) %in% dcm_nf_samples)
    
    tables <- list(count_table_filtered, col_data_filtered)
    
    return (tables)
    
}



In [10]:
d1_final_tables <- create_filtered_tables(d1_col_data, d1_count_table)
d2_final_tables <- create_filtered_tables(d2_col_data, d2_count_table)



d1_final_count_table <- d1_final_tables[[1]]
d2_final_count_table <- d2_final_tables[[1]]


d1_final_col_data  <- d1_final_tables[[2]]
d2_final_col_data  <- d2_final_tables[[2]]


In [12]:
#run DESeq analysis 


run_DESeq <- function(count_table, col_data){
    
    dds <- DESeqDataSetFromMatrix(count_table, col_data, design = ~ condition)
    dds <- DESeq(dds)
    
    return (dds) 
}



In [31]:
#find differentially expressed genes (p_adjusted < 0.05 and log2fold change >=1)


res <- results(d1_dds,contrast=c("condition","Dilated cardiomyopathy (DCM)", "Non-Failing Donor"), alpha = 0.05, lfcThreshold =1)

dim(res)

res <- as_tibble(res)

dim(res)
significant_genes <- res %>% filter(!is.na(res$padj) & padj < 0.05)
upreg_genes <- significant_genes %>% filter(log2FoldChange > 0)
downreg_genes <- significant_genes %>% filter(log2FoldChange < 0)



dim(upreg_genes) #547 genes upregulated in DCM
dim(downreg_genes) #247 genes downregulate in DCM 

[1] 39376     6

[1] 39376     6

[1] 555   6

[1] 247   6

In [13]:
d1_dds <- run_DESeq(d1_final_count_table, d1_final_col_data)

d2_dds <- run_DESeq(d2_final_count_table, d2_final_col_data)

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”


  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R. [This is a message, not a warning or an error]



estimating size factors



  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R. [This is a message, not a warning or an error]



estimating dispersions



gene-wise dispersion estimates



... (5 outputs left out)

-- replacing outliers and refitting for 689 genes
-- DESeq argument 'minReplicatesForReplace' = 7 
-- original counts are preserved in counts(dds)



estimating dispersions



fitting model and testing



  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R. [This is a message, not a warning or an error]



In [42]:
#Get gene annotations to map gene IDs to their gene symbols 

create_gene_annot_table <- function(gene_annot_file)
    
    {
    
    gene_annot_table <- read.table(gene_annot_file, header=TRUE,
                                   sep = "\t", fill=TRUE, quote="")
    
    gene_annot_table_filtered <- gene_annot_table %>% select(GeneID, Symbol)
    
    return (gene_annot_table_filtered)
    
}

,GeneID,Symbol,Description,Synonyms,GeneType,EnsemblGeneID,Status,ChrAcc,ChrStart,ChrStop,Orientation,Length,GOFunctionID,GOProcessID,GOComponentID,GOFunction,GOProcess,GOComponent
,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,100287102,DDX11L1,DEAD/H-box helicase 11 like 1 (pseudogene),,pseudo,ENSG00000290825,active,NC_000001.11,11874,14409,positive,1652,,,,,,
2,653635,WASH7P,"WASP family homolog 7, pseudogene",FAM39F|WASH5P,pseudo,,active,NC_000001.11,14362,29370,negative,1769,,,,,,
3,102466751,MIR6859-1,microRNA 6859-1,hsa-mir-6859-1,ncRNA,ENSG00000278267,active,NC_000001.11,17369,17436,negative,68,,,,,,
4,107985730,MIR1302-2HG,MIR1302-2 host gene,,ncRNA,,active,NC_000001.11,29926,31295,positive,538,,,,,,
5,100302278,MIR1302-2,microRNA 1302-2,MIRN1302-2|hsa-mir-1302-2,ncRNA,ENSG00000284332,active,NC_000001.11,30366,30503,positive,138,,GO:0035195,,,miRNA-mediated gene silencing,
6,645520,FAM138A,family with sequence similarity 138 member A,F379|FAM138F,ncRNA,ENSG00000237613,active,NC_000001.11,34611,36081,negative,1130,,,,,,


In [ ]:
gene_annot_file <- "Human.GRCh38.p13.annot.tsv.gz"
gene_annot_table <- create_gene_annot_table(gene_annot_file)